# autoforecast Energy Walkthrough

This notebook shows the first useful loop: make an energy panel, build strict-past features, validate a DSL feature, compare against a seasonal baseline, and run the promotion gate.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if (ROOT / "src").exists():
    sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd

from autoforecast.features import FeatureSpec, build_feature_matrix
from autoforecast.features.validate import validate_feature_spec
from autoforecast.experiments.metrics import rmse, diebold_mariano_squared_error
from autoforecast.experiments.promote import promotion_gate

## 1. Build a Small Energy Panel

The synthetic target acts like daily peak demand: hot days, cold nights, weekdays, and last week's demand all matter.

In [ ]:
rng = np.random.default_rng(7)
n_days = 730
date = pd.date_range("2024-01-01", periods=n_days, freq="D")
day = np.arange(n_days)

annual = np.sin(2 * np.pi * day / 365.25)
weekly = np.sin(2 * np.pi * day / 7)
tmax = 25 + 8 * annual + 3 * weekly + rng.normal(0, 1.7, n_days)
tmin = tmax - 9 + rng.normal(0, 1.0, n_days)
cdd = np.maximum(tmax - 24, 0)
hdd = np.maximum(15 - tmin, 0)
is_weekend = (date.dayofweek >= 5).astype(float)

demand = 7300 + 120 * cdd + 75 * hdd - 360 * is_weekend + 95 * weekly + rng.normal(0, 95, n_days)
for idx in range(7, n_days):
    demand[idx] = 0.72 * demand[idx] + 0.28 * demand[idx - 7]

panel = pd.DataFrame({
    "region": "NSW1",
    "date": date,
    "peak_demand": demand,
    "tmax": tmax,
    "tmin": tmin,
    "cdd": cdd,
    "hdd": hdd,
})
panel.head()

## 2. Add a DSL Feature

The DSL lets an agent compose features without writing arbitrary Python. Here, it proposes a weekend cooling-load interaction.

In [ ]:
cdd_x_weekend = FeatureSpec.from_dict({
    "op": "interaction",
    "args": [
        {"op": "raw", "args": ["cdd"]},
        {"op": "raw", "args": ["is_weekend"]},
    ],
})

validation = validate_feature_spec(
    panel,
    spec=cdd_x_weekend,
    name="cdd_x_weekend",
    date_col="date",
    target_col="peak_demand",
    id_cols=["region"],
)
validation

## 3. Build Strict-Past Features

Lag and rolling features use only information available before the forecast target day.

In [ ]:
features = [
    "lag_1", "lag_7", "rolling_7_mean", "rolling_28_mean",
    "day_of_week", "is_weekend", "tmax", "cdd", "hdd",
]

matrix = build_feature_matrix(
    panel,
    date_col="date",
    target_col="peak_demand",
    id_cols=["region"],
    features=features,
    specs={"cdd_x_weekend": cdd_x_weekend},
)
matrix.tail()

## 4. Compare Against Seasonal Naive

This tiny model is not the final modeling story. It is just enough to prove the experiment spine works.

In [ ]:
feature_cols = features + ["cdd_x_weekend"]
split_date = matrix["date"].max() - pd.Timedelta(days=90)
train = matrix[matrix["date"] <= split_date].dropna(subset=feature_cols + ["peak_demand"])
holdout = matrix[matrix["date"] > split_date].dropna(subset=feature_cols + ["peak_demand"])

x_train = np.column_stack([np.ones(len(train)), train[feature_cols].to_numpy(float)])
y_train = train["peak_demand"].to_numpy(float)
coef, *_ = np.linalg.lstsq(x_train, y_train, rcond=None)

x_holdout = np.column_stack([np.ones(len(holdout)), holdout[feature_cols].to_numpy(float)])
pred = x_holdout @ coef
baseline = holdout["lag_7"].to_numpy(float)
actual = holdout["peak_demand"].to_numpy(float)

baseline_rmse = rmse(actual, baseline)
model_rmse = rmse(actual, pred)
dm = diebold_mariano_squared_error(actual, pred, baseline)
decision = promotion_gate(candidate_metric=model_rmse, baseline_metric=baseline_rmse)

{
    "baseline_rmse": baseline_rmse,
    "model_rmse": model_rmse,
    "dm_p_value": dm.p_value,
    "promote": decision.accepted,
    "reason": decision.reason,
}